# Sinh real/fake theo từng part trên Kaggle
Mỗi lượt chọn **một part clip sạch**. Chỉnh PART, đường dẫn và thông số bên dưới, rồi chạy lần lượt các cell.
Mặc định mỗi part chia độc lập 80/10/10 theo nhóm người/nguồn; không cần split-lock của part trước.
Bật Internet/GPU. Worker MuseTalk chỉ được cài khi danh sách generator có musetalk_1_5.
Cần pilot thật để xác nhận checkpoint/worker trên Kaggle; hiện kiểm thử dùng media/generator fixture.


In [ ]:
from pathlib import Path
import subprocess, sys, os
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
REPO_URL = "https://github.com/linhxm/vn-av-forensics.git"  # Sửa nếu đổi repository.
ROOT = Path("/kaggle/working/vn-av-df-forensics")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)
assert (ROOT / "settings.py").is_file(), "Cần upload/push code mới trước khi dùng notebook"
sys.path.insert(0, str(ROOT))
subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(ROOT / "data_pipeline"), "-e", str(ROOT) + "[generation]"], check=True)
import settings
settings.bootstrap()
from vn_av_df.actions import execute


## 1. Chọn part và thông số
CSV nguồn không dùng ở bước này: đầu vào là part clip sạch đã review/export.
CLIPS_PER_SPLIT=2 để chạy thử; 0 để lấy mọi parent đủ điều kiện. Số output lớn hơn số parent vì có real/full/partial và có thể sham.
Plan đã tạo sẽ khóa cấu hình. Đổi cấu hình sau đó cần plan/output mới, không sửa part đã chốt tại chỗ.


In [ ]:
PART = 1  # Ví dụ: 2 -> xử lý vn-av-df-data-part2.
GENERATION_NAME = f"vn-av-df-data-part{PART}"
CLEAN_DATASET = Path("/kaggle/input/YOUR_CLEAN_PART") / GENERATION_NAME
SPLIT_RATIOS = {"train": 0.8, "validation": 0.1, "test": 0.1}  # Tỷ lệ clip sạch; giữ nguyên nhóm.
SPLIT_HISTORY = []  # Tạm tắt lịch sử giữa các part, kể cả PART >= 2.
# Bật lại: liệt kê các split-lock bao phủ mọi part trước; file tích lũy đủ thì chỉ cần một.
# SPLIT_HISTORY = [Path("/kaggle/input/PART1_LOCK/split-lock.json")]  # Ví dụ cho part 2.
DEVICE = "cuda"
SEED = 42  # Ngẫu nhiên của split/plan; giữ cố định giữa các part.
CLIPS_PER_SPLIT = 2  # 2 = pilot; 0 = toàn bộ clip đủ điều kiện trong từng split.
PARTIAL_SECONDS = [0.4, 0.8, 1.6, 2.4]  # Chọn ngẫu nhiên 1 độ dài fake cục bộ/parent.
CRF = 18                # Nén H.264 chung cho real/fake; số nhỏ hơn = file lớn hơn.
MAX_SIDE = 640          # Giới hạn cạnh dài của video output.
# Thiết kế 2×2. donor: miệng vẽ theo tiếng thật khác của cùng người (mối đe dọa chính).
# source: miệng vẽ theo đúng tiếng gốc (chỉ có artifact), đối chứng ở validation/test.
# Parent không tìm được donor cùng speaker_id bị bỏ qua và ghi vào plan.
FAKE_AUDIO_MODES_BY_SPLIT = {
    "train": ["donor"],
    "validation": ["donor", "source"],
    "test": ["donor", "source"],
}
INCLUDE_SHAM = True     # Sham: ghép tiếng thật cùng người, không AI (nhãn AI=0, có đoạn lệch A/V).
GENERATORS_BY_SPLIT = {
    "train": ["wav2lip_gan"],
    "validation": ["wav2lip_gan"],
    "test": ["wav2lip_gan", "musetalk_1_5"],
}
# Pilot chỉ Wav2Lip, không cài MuseTalk:
# GENERATORS_BY_SPLIT = {s: ["wav2lip_gan"] for s in ("train", "validation", "test")}
MUSETALK_BATCH_SIZE = 4  # Giảm nếu worker MuseTalk thiếu VRAM.
MUSETALK_SEED = 42


In [ ]:
settings.PART, settings.GENERATION_NAME = PART, GENERATION_NAME
settings.CLEAN_DATASET, settings.SPLIT_HISTORY = CLEAN_DATASET, SPLIT_HISTORY
settings.SPLIT_RATIOS = SPLIT_RATIOS
settings.DEVICE = DEVICE
cfg = settings.config()
cfg["seed"] = SEED
cfg["generation"].update(
    clips_per_split=CLIPS_PER_SPLIT, partial_seconds=PARTIAL_SECONDS,
    crf=CRF, max_side=MAX_SIDE, fake_audio_modes_by_split=FAKE_AUDIO_MODES_BY_SPLIT,
    include_sham=INCLUDE_SHAM, generators_by_split=GENERATORS_BY_SPLIT,
)
cfg["musetalk"].update(batch_size=MUSETALK_BATCH_SIZE, seed=MUSETALK_SEED)
print("Part:", PART, "| Input:", cfg["clean_dataset"], "| Output:", cfg["generated_dataset"])
print("Thông số:", cfg["generation"])
print("Tỷ lệ split:", cfg["split_ratios"], "| Lịch sử:", cfg["split_history"])
execute("validate", cfg)
from vn_av_df.common.runtime import read_json
from vn_av_df.generation import validate_plan_settings
if Path(cfg["plan"]).exists():
    plan = read_json(cfg["plan"])
    validate_plan_settings(cfg, plan)
else:
    execute("plan", cfg)
execute("generator_setup", cfg)


In [ ]:
needs_musetalk = any("musetalk_1_5" in names for names in cfg["generation"]["generators_by_split"].values())
if needs_musetalk:
    subprocess.run([sys.executable, str(ROOT / "environments/bootstrap_worker.py"), "musetalk", "--root", str(ROOT)], check=True)
    worker_python = str(ROOT / ".venv-musetalk/bin/python")
    settings.MUSETALK_PYTHON = worker_python
    cfg["musetalk"]["python"] = worker_python
    print("Worker ready:", worker_python)
else:
    print("Không chọn MuseTalk: bỏ qua cài worker MuseTalk.")


## 2. Sinh video và tải part về local
Generation ghi record để tiếp tục nếu bị ngắt; chạy lại cùng plan/output. Không tự đổi generator khi có lỗi.
Output còn pending, chưa dùng để train cho đến khi review/finalize.


In [ ]:
execute("generate", cfg)


In [ ]:
import shutil, zipfile
folder = Path(cfg["generated_dataset"])
archive = shutil.make_archive("/kaggle/working/" + folder.name, "zip", folder.parent, folder.name)
print("Tải part:", archive)
if needs_musetalk:
    with zipfile.ZipFile("/kaggle/working/generation-environment.zip", "w", zipfile.ZIP_DEFLATED) as z:
        for p in (ROOT / ".kaggle-tools/musetalk").glob("*"):
            if p.is_file():
                z.write(p, p.name)
    print("Tải thêm generation-environment.zip để giữ log môi trường.")


## 3. Review/finalize ở local
Giải nén ZIP vào datasets/vn-av-df-data/, được folder vn-av-df-data-partN.
Đặt PART=N trong data_settings.py rồi chạy generation/04_review.py → 05_finalize.py → 07_export.py.
Upload ZIP đã finalize sang Kaggle training; chọn part đó hoặc gộp các part trong train.ipynb.
split-lock.json vẫn ghi lại cách chia part; mặc định không cần đưa sang part kế tiếp. Giữ file để tái lập hoặc bật lại lịch sử sau này.
